### Cartesian radiating sphere test

Flux factor $f = F_r / E$ and Eddington factor $p = P_{rr} / E$ from the last mesh snapshot along the outward $+x$ ray from the sphere center, compared with the analytic radiating-sphere solution.

In [ ]:
import os
import sys
from pathlib import Path
import glob

import matplotlib.pyplot as plt
import h5py
import numpy as np

try:
    here = Path(__file__).resolve().parent
except NameError:
    here = Path.cwd()

sys.path.append(str((here / '..' / 'plot_functions').resolve()))

import plot_functions as pf

plots_dir = here / 'plots'
plots_dir.mkdir(exist_ok=True)

Simulation and test parameters

In [ ]:
# Directory with mesh_plt*.h5 dumps (only the last dump is used below)
directory_path = (
    '/pscratch/sd/b/bigqke/bigqke_scratch/emu_methods_paper_simulations/bigqke_per/radiating_sphere_test/'
    'radiating_sphere_test_L_cm_4.212e6_2.808e6_2.808e6_ncell_96_64_64_dts_3.66e-7_deposition_1_bc_periodic_empty_ppepc_378'    
)
Nx, Ny, Nz = 96, 64, 64 # Cartesian grid size; checked against the last dump
r_xi, r_yi, r_zi = 32, 32, 32 # Sphere-center cell index

# directory_path = (
#     '/pscratch/sd/b/bigqke/bigqke_scratch/emu_methods_paper_simulations/bigqke_per/radiating_sphere_test/'
#     'radiating_sphere_test_L_cm_4.212e6_2.808e6_2.808e6_ncell_192_128_128_dts_1.83e-7_deposition_1_bc_periodic_empty_ppepc_378_stored'    
# )
# Nx, Ny, Nz = 192, 128, 128 # Cartesian grid size; checked against the last dump
# r_xi, r_yi, r_zi = 64, 64, 64 # Sphere-center cell index

# directory_path = (
#     '/pscratch/sd/b/bigqke/bigqke_scratch/emu_methods_paper_simulations/bigqke_per/radiating_sphere_test/'
#     # 'radiating_sphere_test_L_cm_3.627e6_2.457e6_2.457e6_ncell_128_128_128_dts_1.60e-7_deposition_1_bc_periodic_empty_ppepc_378_stored'
#     # 'radiating_sphere_test_L_cm_3.627e6_2.457e6_2.457e6_ncell_128_128_128_dts_1.60e-7_deposition_1_bc_outflow_ppepc_92_stored'
#     'radiating_sphere_test_L_cm_3.627e6_2.457e6_2.457e6_ncell_128_128_128_dts_1.60e-7_deposition_1_bc_periodic_empty_ppepc_92_stored'
#     ''
# )
# r_xi, r_yi, r_zi = 45, 64, 64 
# Nx, Ny, Nz = 128, 128, 128

directory_path = (
    '/pscratch/sd/b/bigqke/bigqke_scratch/emu_methods_paper_simulations/bigqke_per/radiating_sphere_test/'
    'radiating_sphere_test_L_cm_4.212e6_2.808e6_2.808e6_ncell_384_256_256_dts_9.15e-8_deposition_1_bc_periodic_empty_ppepc_378_stored'
)
Nx, Ny, Nz = 384, 256, 256 # Cartesian grid size; checked against the last dump
r_xi, r_yi, r_zi = 128, 128, 128 # Sphere-center cell index

###################################
# changing number of particles per cell

directory_path = (
    '/pscratch/sd/b/bigqke/bigqke_scratch/emu_methods_paper_simulations/bigqke_per/radiating_sphere_test/'
    'radiating_sphere_test_L_cm_4.212e6_2.808e6_2.808e6_ncell_384_256_256_dts_9.15e-8_deposition_1_bc_periodic_empty_ppepc_378_stored'
    # 'radiating_sphere_test_L_cm_4.212e6_2.808e6_2.808e6_ncell_384_256_256_dts_9.15e-8_deposition_1_bc_periodic_empty_ppepc_212'
    # 'radiating_sphere_test_L_cm_4.212e6_2.808e6_2.808e6_ncell_384_256_256_dts_9.15e-8_deposition_1_bc_periodic_empty_ppepc_92'
)
Nx, Ny, Nz = 384, 256, 256 # Cartesian grid size; checked against the last dump
r_xi, r_yi, r_zi = 128, 128, 128 # Sphere-center cell index

# Physics / comparison parameters (same as the 1D spherical test, except r_min_over_R).
# The 1D script cuts at 0.6; here we score the whole ray from the center outward.
R = 1.17e6          # sphere radius, cm
kappa_int_invcm = 0.00000854544806011998 # electron neutrino absorption opacity, 1/cm
kappa_ext_invcm = 1.4877457720665111e-9 # electron neutrino absorption opacity, 1/cm
kappa_R = kappa_int_invcm * R # radial optical depth used in the analytic path length
n_mu = 2000         # number of mu bins for the analytic moment integrals
tolerance = 0.07    # max allowed absolute error in f and p
r_min_over_R = 0.0  # inner radius cutoff for the error test (include all r >= 0)

# plots
limlow = None
limup = -1

Read the last mesh dump and extract the outward $+x$ ray

In [ ]:
# Sort by the integer plot index in mesh_pltNNNNN.h5 so [-1] is the final dump.
filenames = sorted(
    glob.glob(os.path.join(directory_path, 'mesh_plt*.h5')),
    key=lambda x: int(os.path.basename(x).split('plt')[1].split('.')[0]),
)

if len(filenames) == 0:
    raise FileNotFoundError(f'No mesh_plt*.h5 files found in {directory_path}')

if not (0 <= r_xi < Nx and 0 <= r_yi < Ny and 0 <= r_zi < Nz):
    raise ValueError(
        f'Sphere-center cell ({r_xi}, {r_yi}, {r_zi}) is outside '
        f'grid ({Nx}, {Ny}, {Nz})'
    )

# Use only the last snapshot (plot index matches nsteps = 508 for this run).
filename = filenames[-1]

with h5py.File(filename, 'r') as hf:
    file_Nx = int(np.array(hf['Nx']))
    file_Ny = int(np.array(hf['Ny']))
    file_Nz = int(np.array(hf['Nz']))
    if (file_Nx, file_Ny, file_Nz) != (Nx, Ny, Nz):
        raise ValueError(
            f'{os.path.basename(filename)} has grid '
            f'({file_Nx}, {file_Ny}, {file_Nz}), expected ({Nx}, {Ny}, {Nz})'
        )

    # Mesh moments: N00_Re = number density; along +x, Fx and Pxx are F_r and P_rr.
    E = np.array(hf['N00_Re(1|ccm)'], dtype=float)
    Fx = np.array(hf['Fx00_Re(1|ccm)'], dtype=float)
    Pxx = np.array(hf['Pxx00_Re(1|ccm)'], dtype=float)

    if E.shape != (Nx, Ny, Nz):
        raise ValueError(
            f'{os.path.basename(filename)} field shape {E.shape} '
            f'does not match ({Nx}, {Ny}, {Nz})'
        )

    dx = float(np.array(hf['dx(cm)']))
    dy = float(np.array(hf['dy(cm)']))
    dz = float(np.array(hf['dz(cm)']))

    # Outward +x ray through the sphere-center cell (domain origin at x = 0).
    E_ray = E[r_xi:, r_yi, r_zi].copy()
    Fr_ray = Fx[r_xi:, r_yi, r_zi].copy()
    Prr_ray = Pxx[r_xi:, r_yi, r_zi].copy()

# Cell centers along x; r = 0 at the sphere-center cell center.
x_centers = (np.arange(Nx) + 0.5) * dx
r_centers = x_centers[r_xi:] - x_centers[r_xi]

print(f'Read last dump {os.path.basename(filename)} from {directory_path}')
print(f'Grid (Nx, Ny, Nz) = ({Nx}, {Ny}, {Nz})')
print(f'dx, dy, dz = {dx:.6g}, {dy:.6g}, {dz:.6g} cm')
print(f'Sphere-center cell (r_xi, r_yi, r_zi) = ({r_xi}, {r_yi}, {r_zi})')
print(f'Ray length = {len(r_centers)} cells along +x')

Using the last snapshot of the simulation to compute flux factor and Eddington factor

In [ ]:
# Non-positive density on the ray becomes NaN so empty cells drop out of the error mask.
E_ray[E_ray <= 0.0] = np.nan

# Constants cancel in the ratios: f = F_r / E, p = P_rr / E from the last dump only.
f_simulation = Fr_ray / E_ray
p_simulation = Prr_ray / E_ray

Analytic radiating-sphere solution

In [ ]:
# Analytic f and p at each cell center along the ray (same formulas as radiating_sphere_test.py).
f_analytic = np.zeros(len(r_centers))
p_analytic = np.zeros(len(r_centers))

for i, ri in enumerate(r_centers):
    if ri < R:
        # Inside: all mu in [-1, 1]; s is path length back to the surface in units of R.
        mu_arr = np.linspace(-1.0, 1.0, n_mu)
        g = np.sqrt(np.maximum(1.0 - (ri / R)**2 * (1.0 - mu_arr**2), 0.0))
        s = ri * mu_arr / R + g
    else:
        # Outside: only mu > sqrt(1 - (R/r)^2) intersects the sphere; s = 2g is the chord.
        mu_c = np.sqrt(1.0 - (R / ri)**2)
        mu_arr = np.linspace(mu_c, 1.0, n_mu)
        g = np.sqrt(np.maximum(1.0 - (ri / R)**2 * (1.0 - mu_arr**2), 0.0))
        s = 2 * g

    # Specific intensity / B = 1 - exp(-kappa_R * s); B and azimuth cancel in the ratios.
    F_mu = 1.0 - np.exp(-kappa_R * s)

    E_int = np.trapezoid(F_mu, mu_arr)
    f_analytic[i] = np.trapezoid(mu_arr * F_mu, mu_arr) / E_int
    p_analytic[i] = np.trapezoid(mu_arr**2 * F_mu, mu_arr) / E_int

Error estimate (last snapshot vs analytic)

In [ ]:
# Compare last-snapshot f and p to the analytic solution where E is finite and r is outside the cutoff.
valid = np.isfinite(f_simulation) & (r_centers >= r_min_over_R * R)

f_error = np.abs(f_simulation[valid] - f_analytic[valid])
p_error = np.abs(p_simulation[valid] - p_analytic[valid])

f_max_error = np.max(f_error)
p_max_error = np.max(p_error)

r_valid = r_centers[valid]

print(
    f'Last dump {os.path.basename(filename)} '
    f'for the region r/R >= {r_min_over_R}'
)
print(
    f'Max error in f: {f_max_error}  '
    f'at r/R = {r_valid[np.argmax(f_error)] / R:.4f}'
)
print(
    f'Max error in p: {p_max_error}  '
    f'at r/R = {r_valid[np.argmax(p_error)] / R:.4f}'
)

if f_max_error < tolerance and p_max_error < tolerance:
    print(f'SUCCESS (errors below tolerance {tolerance})')
else:
    print(f'FAIL (errors above tolerance {tolerance})')

Plot simulation vs analytic $f$ and $p$

In [ ]:
# Profile of last-snapshot and analytic f, p versus r/R along the +x ray.
r_over_R = r_centers / R

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(r_over_R[limlow:limup], f_simulation[limlow:limup], color='C0', linestyle='dotted', linewidth=4, label=r'$\hat{\mathrm{f}}_{\mathrm{sim}}$')
ax.plot(r_over_R[limlow:limup], f_analytic[limlow:limup], color='C0', linestyle='solid', alpha=0.5, linewidth=4, label=r'$\hat{\mathrm{f}}_{\mathrm{analytic}}$')
ax.plot(r_over_R[limlow:limup], p_simulation[limlow:limup], color='C1', linestyle='dotted', linewidth=4, label=r'$\hat{\mathrm{p}}_{\mathrm{sim}}$')
ax.plot(r_over_R[limlow:limup], p_analytic[limlow:limup], color='C1', linestyle='solid', alpha=0.5, linewidth=4, label=r'$\hat{\mathrm{p}}_{\mathrm{analytic}}$')
ax.axvline(1.0, color='k', linestyle=':', linewidth=1.5, label=r'$r = R$')
ax.axvline(r_min_over_R, color='gray', linestyle=':', linewidth=1.5,
           label=rf'$r/R = {r_min_over_R}$')
ax.set_xlabel(r'$r / R$')
ax.set_ylabel(r'$f,\, p$')
ax.set_xlim(None, None)
leg = ax.legend(framealpha=0.0, ncol=1, fontsize=16)
pf.apply_custom_settings(ax, leg, False)
outfile = plots_dir / f"{directory_path.rstrip('/').split('/')[-1]}_f_p_vs_r.pdf"
fig.savefig(outfile, bbox_inches='tight')
plt.show()
plt.close(fig)
print(f'Saved {outfile}')

Absolute error versus $r/R$

In [ ]:
# Absolute |f_sim - f_analytic| and |p_sim - p_analytic| on the valid ray points, with the tolerance line.
fig, ax = plt.subplots(figsize=(10, 6))
# ax.plot(r_over_R[valid][limlow:limup], f_error[limlow:limup], color='C0', linewidth=4, label=r'$|f_{\mathrm{sim}} - f_{\mathrm{analytic}}|$')
# ax.plot(r_over_R[valid][limlow:limup], p_error[limlow:limup], color='C1', linewidth=4, label=r'$|p_{\mathrm{sim}} - p_{\mathrm{analytic}}|$')
ax.plot(r_over_R[valid][limlow:limup], np.log(f_error[limlow:limup]), color='C0', linewidth=4, label=r'$\log\,|f_{\mathrm{sim}} - f_{\mathrm{analytic}}|$')
ax.plot(r_over_R[valid][limlow:limup], np.log(p_error[limlow:limup]), color='C1', linewidth=4, label=r'$\log\,|p_{\mathrm{sim}} - p_{\mathrm{analytic}}|$')

print(f'r_over_R = {list(r_over_R[valid][limlow:limup])}')
print(f'f_error = {list(f_error[limlow:limup])}')
print(f'p_error = {list(p_error[valid][limlow:limup])}')

ax.axhline(np.log(tolerance), color='k', linestyle='--', linewidth=1.5,
           label=rf'tolerance $= {tolerance}$')
ax.axvline(1.0, color='k', linestyle=':', linewidth=1.5, label=r'$r = R$')
ax.set_xlabel(r'$r / R$')
ax.set_ylabel(r'$\log(\, \mathrm{Absolute\,error}\,)$')
ax.set_ylim(-10, -1)
ax.set_xlim(0.0, 2.0)
leg = ax.legend(framealpha=0.0, ncol=1, fontsize=16)
pf.apply_custom_settings(ax, leg, False)
outfile = plots_dir / f"{directory_path.rstrip('/').split('/')[-1]}_f_p_error_vs_r.pdf"
fig.savefig(outfile, bbox_inches='tight')
plt.show()
plt.close(fig)
print(f'Saved {outfile}')